In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# import packages
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import time
import logging
from logging.handlers import TimedRotatingFileHandler
from pathlib import Path
from functools import wraps

import yaml
from types import SimpleNamespace

In [ ]:
# config path
CONFIG_PATH = "/content/drive/MyDrive/Progress/Demand Planning/configs/config.yaml"

In [ ]:
# load config file
def load_config(path):
    with open(path, "r") as f:
        config_dict = yaml.safe_load(f)

    return config_dict

config = load_config(CONFIG_PATH)

In [ ]:
def setup_logger():
    logger = logging.getLogger("DataPrepPipeline")
    logger.setLevel(logging.INFO)

    # Avoid adding duplicate handlers
    if logger.handlers:
        return logger
    log_path = Path(config["data"]["log_path"])
    log_path.parent.mkdir(parents=True, exist_ok=True)

    # File handler (append mode)
    file_handler = TimedRotatingFileHandler(
      log_path,
      when="midnight",
      interval=1,
      backupCount=30,  # keep last 30 days
      encoding="utf-8")

    file_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | "
        "%(filename)s:%(lineno)d | "
        "%(funcName)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    return logger

logger = setup_logger()

In [ ]:
def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        try:
            logger.info(f"🚀 Starting: {func.__name__}")
            result = func(*args, **kwargs)
            duration = (time.time() - start) / 60
            logger.info(f"✅ Completed: {func.__name__} in {duration:.2f} mins")
            return result

        except Exception as e:
            logger.exception(f"❌ Error in {func.__name__}: {str(e)}")
            raise

    return wrapper

In [ ]:
class DataIngestion:
    def __init__(self, config):
        self.config = config

    @log_time
    def load_items(self):
        df = pd.read_csv(self.config['data']['items_path'])
        logger.info(f"Items loaded: {df.shape}")
        return df

    @log_time
    def load_stores(self):
        df = pd.read_csv(self.config['data']['stores_path'])
        logger.info(f"Stores loaded: {df.shape}")
        return df.rename(columns={"type": "store_type"})

    @log_time
    def load_transactions(self):
        df = pd.read_csv(self.config['data']['transactions_path'], parse_dates=["date"])
        logger.info(f"Transactions loaded: {df.shape}")
        return df

    @log_time
    def load_oil(self):
        df = pd.read_csv(self.config['data']['oil_path'], parse_dates=["date"])
        logger.info(f"Oil data loaded: {df.shape}")
        return df

    @log_time
    def load_holidays(self):
        df = pd.read_csv(
            self.config['data']['holiday_path'],
            usecols=["date", "type", "locale"],
            parse_dates=["date"],
        )

        df = df.groupby("date").agg({"type": "first", "locale": "first"}).reset_index()
        logger.info(f"Holidays aggregated: {df.shape}")

        return df.rename(columns={"type": "holiday_type"})

In [ ]:
class DataFilter:
    def __init__(self, config):
        self.config = config

    @log_time
    def filter_train(self, unique_items):
        unique_items = set(unique_items)

        reader = pd.read_csv(
            self.config['data']['train_path'],
            dtype={
                "store_nbr": "int16",
                "item_nbr": "int32",
                "unit_sales": "float32",
                "onpromotion": "boolean",
            },
            usecols=["date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"],
            parse_dates=["date"],
            chunksize=self.config['processing']['chunk_size'],
        )

        dfs = []
        total_rows = 0
        total_chunks = 0

        for chunk in tqdm(reader, desc="Processing chunks"):
            total_chunks += 1
            filtered = chunk[chunk["item_nbr"].isin(unique_items)]

            total_rows += len(filtered)

            if not filtered.empty:
                dfs.append(filtered)

        logger.info(f"Total chunks processed: {total_chunks}")
        logger.info(f"Total rows kept: {total_rows:,}")

        df = pd.concat(dfs, ignore_index=True)

        logger.info(f"Filtered dataframe shape: {df.shape}")

        df.to_parquet(self.config['data_prep']['filtered_path'])
        logger.info(f"Saved filtered data to {self.config['data_prep']['filtered_path']}")

        return df

In [ ]:
class DataMerger:
    def __init__(self, config):
        self.config = config

    @log_time
    def merge_all(self, core_df, items_df, stores_df, tx_df, oil_df, holiday_df):

        logger.info("Starting merge operations")

        df = core_df.merge(items_df, on="item_nbr", how="inner")
        logger.info(f"After items merge: {df.shape}")

        df = df.merge(stores_df, on="store_nbr", how="inner")
        logger.info(f"After stores merge: {df.shape}")

        df = df.merge(tx_df, on=["date", "store_nbr"], how="inner")
        logger.info(f"After transactions merge: {df.shape}")

        df = df.merge(oil_df.drop_duplicates("date"), on="date", how="left")
        logger.info(f"After oil merge: {df.shape}")

        df = df.merge(holiday_df, on="date", how="left")
        logger.info(f"After holiday merge: {df.shape}")

        return df

In [ ]:
class DataPreparer:
    def __init__(self, config):
        self.config = config

    @log_time
    def filter_horizon(self, df):
        max_date = df["date"].max()

        df = df[
            df["date"]
            >= (max_date - pd.DateOffset(years=self.config['processing']['train_horizon_years']))
        ]

        logger.info(f"After horizon filter: {df.shape}")

        return df

In [ ]:
class DataPreparationPipeline:
    def __init__(self, config):
        self.config = config

        self.ingestion = DataIngestion(config)
        self.filter = DataFilter(config)
        self.merger = DataMerger(config)
        self.preparer = DataPreparer(config)

    def run(self):
        total_start = time.time()

        try:
            logger.info("========== PIPELINE START ==========")

            # Load data
            items_df = self.ingestion.load_items()
            stores_df = self.ingestion.load_stores()
            tx_df = self.ingestion.load_transactions()
            oil_df = self.ingestion.load_oil()
            holiday_df = self.ingestion.load_holidays()

            # Filter category
            cat = self.config['processing']['category']
            items_subset = items_df[items_df["family"] == cat]

            logger.info(f"Category '{cat}' items: {items_subset.shape}")

            unique_items = items_subset["item_nbr"].unique()

            # Filter large dataset
            core_df = self.filter.filter_train(unique_items)

            # Merge
            df = self.merger.merge_all(
                core_df,
                items_subset,
                stores_df,
                tx_df,
                oil_df,
                holiday_df,
            )

            # Final filter
            df = self.preparer.filter_horizon(df)

            # Save final
            df.to_parquet(self.config['data_prep']['final_path'])
            logger.info(f"Final dataset saved: {self.config['data_prep']['final_path']}")

            total_time = (time.time() - total_start) / 60
            logger.info(f"🚀 TOTAL PIPELINE TIME: {total_time:.2f} mins")

            return df

        except Exception as e:
            logger.exception("❌ PIPELINE FAILED")
            raise

In [ ]:
pipeline = DataPreparationPipeline(config)
final_df = pipeline.run()
final_df.head()